# Baseline implicit ALS trên Silver

Thử nhanh trên **mẫu user Video Games**, so sánh Popularity và ALS bằng Recall/NDCG@10/20.
Giữ toàn bộ lịch sử của user được chọn, chia theo thời gian, không lọc rating ≥4.
**Chỉ một kernel chạy Spark; dừng EDA/Silver/batch trước khi chạy.**

Notebook dùng module `als_baseline.py`; không sửa Silver. Xem `../docs/als-baseline.md`.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

NOTEBOOKS = Path('/workspace/notebooks')
if str(NOTEBOOKS) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS))
from als_baseline import run_baseline


## 1. Khóa cấu hình

Mặc định 5.000 user, seed 42, rank 16, 5 iterations. Chọn user bằng hash ID; giữ mọi
interaction, binary distinct pairs. Không k-core. Candidates = items đã thấy trong train của mẫu, có repeat.
Validation: train <2021, first-next trong [2021,2022). `evaluate_test=False` giữ kín test.
Chỉ bật test sau khi khóa cấu hình: refit <2022, first-next trong [2022,2023-10-01).


In [ ]:
CONFIG = dict(
    category='Video_Games',
    silver_run='run_20261016T022209697481Z',
    sample_size=5000,
    seed=42,
    rank=16,
    max_iter=5,
    reg_param=0.1,
    alpha=1.0,
    evaluate_test=False,
    min_free_gb=20,
)
# Điền run directory cũ để xem kết quả mà không train lại.
REPORT_DIRECTORY = None
CONFIG


## 2. Chạy pipeline

Kiểm tra manifest/schema/counts Silver → lấy mẫu → split/targets → mapping train-only
→ Popularity → implicit ALS → Top-K → đánh giá → lưu artifact. In tiến độ từng pha.
Session Spark local[2] được dừng trong finally, kể cả khi có lỗi. Không tải model hoặc dependency mới.


In [ ]:
if REPORT_DIRECTORY is None:
    REPORT_DIRECTORY = run_baseline(**CONFIG)
report_path = Path(REPORT_DIRECTORY)
run = json.loads((report_path / 'run.json').read_text(encoding='utf-8'))
if run['status'] != 'complete':
    raise RuntimeError(f"Run chưa complete: {run.get('error', run['status'])}")
print('Report:', report_path)
display(pd.DataFrame([{key: run.get(key) for key in (
    'status', 'category', 'sampled_users', 'sampled_events', 'source_reviews', 'prepare_seconds'
)}]))


## 3. Population và tài nguyên

Counts là của **mẫu pilot**, không thay số liệu toàn category. Target ngoài candidates
vẫn giữ ở population. Cold user được đánh giá riêng và dùng Popularity khi có fallback.


In [ ]:
population = []
for fold, result in run['folds'].items():
    population.append(dict(fold=fold, train_pairs=result['train_pairs'],
        train_users=result['train_users'], train_items=result['train_items'],
        **result['population'], fallback_queries=result['fallback_queries'],
        fit_seconds=result['fit_seconds'], recommendation_seconds=result['recommendation_seconds'],
        evaluation_seconds=result['evaluation_seconds']))
display(pd.DataFrame(population))
resources = json.loads((report_path / 'resources.json').read_text(encoding='utf-8'))
display(pd.DataFrame(resources['samples']))


## 4. So sánh baseline

`all_queries`: mọi query có nhãn, kể cả cold user/item; không có gợi ý tính miss.
`warm_full_targets`: user có history, tất cả targets nằm trong candidates.
Popularity dùng cùng train/candidates; `als_popularity_fallback` chỉ fallback cho cold user.
Không lấy RMSE làm metric chính của bài toán xếp hạng.


In [ ]:
metrics = pd.DataFrame([dict(fold=fold, **row)
    for fold, result in run['folds'].items() for row in result['metrics']])
display(metrics.sort_values(['fold', 'cohort', 'k', 'model']).reset_index(drop=True))


## 5. Xem Top-K và quyết định lượt tiếp theo

Chỉ hiển thị tối đa 5 user đã hash và item IDs. Mappings/model/labels lưu trong thư mục run,
không commit dữ liệu user vào Git. Ghi nhận ALS tốt hơn hay kém Popularity trên cùng cohort;
chưa suy kết quả mẫu thành toàn category. Có thể tăng lên 50.000 user sau khi kiểm tra
thời gian/RAM và số query warm; tuning chỉ trên validation.


In [ ]:
for fold, result in run['folds'].items():
    print(fold)
    display(pd.DataFrame(result['preview']))
